# CDEA Contrastive Quickstart

Run CDEA in notebook and switch base evidence between Grad-CAM and Integrated Gradients.

In [7]:
from pathlib import Path
import sys
import torch
import torch.nn.functional as F

REPO = Path('/Users/ssuresh/gambit')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from core.runner import CDEAExplainer
from core.hypotheses import TopMSelector
from core.device import get_device
from core.interaction import get_interaction
from modality.grid_regions import VisionGridUnitSpace
from instantiations.contrastive.objective import ContrastiveObjective
from instantiations.contrastive.allocator import OptimizationAllocator
from base_evidence.gradcam_regions import GradCAMRegionsProvider
from base_evidence.integrated_gradients_regions import IntegratedGradientsRegionsProvider
from examples.contrastive_explanation import TV_INPUT_SIZE, _load_batch, get_torchvision_model, visualize_contrastive

In [8]:
dataset = 'cifar10'
batch_size = 4
model_name = 'resnet18'
pretrained = False
num_alloc_steps = 25
evidence_mode = 'gradcam'  # 'gradcam' or 'ig'

interaction_mode = 'none'
interaction_dim = 64
interaction_heads = 4
interaction_ffn = 256
interaction_dropout = 0.0

ig_steps = 24
ig_baseline = 'zero'

use_shared = True
lambda_partition = 0.1
grid_h, grid_w = 7, 7

In [9]:
device = get_device()
x, class_names, num_classes = _load_batch(dataset, batch_size, device)
if x.shape[-2:] != (TV_INPUT_SIZE, TV_INPUT_SIZE):
    x = F.interpolate(x, size=(TV_INPUT_SIZE, TV_INPUT_SIZE), mode='bilinear', align_corners=False)

model = get_torchvision_model(model_name, num_classes, pretrained=pretrained).to(device).eval()
top_k = min(5, num_classes)
embed_dim = interaction_dim if interaction_mode != 'none' else None

unit_space = VisionGridUnitSpace(grid_h, grid_w, baseline='blur', embed_dim=embed_dim)
selector = TopMSelector(m=top_k)
if evidence_mode == 'gradcam':
    base_evidence = GradCAMRegionsProvider(grid_h, grid_w)
elif evidence_mode == 'ig':
    base_evidence = IntegratedGradientsRegionsProvider(grid_h, grid_w, steps=ig_steps, baseline=ig_baseline)
else:
    raise ValueError('evidence_mode must be gradcam or ig')

interaction = get_interaction(
    flag=interaction_mode,
    d_model=(interaction_dim if interaction_mode != 'none' else None),
    num_heads=interaction_heads,
    dim_feedforward=interaction_ffn,
    dropout=interaction_dropout,
)
objective = ContrastiveObjective(lambda_suff=1.0, lambda_margin=1.0, lambda_sparse=0.05, lambda_overlap=0.2)
allocator = OptimizationAllocator(
    objective=objective,
    num_steps=num_alloc_steps,
    lr=0.2,
    use_shared=use_shared,
    lambda_disjoint=0.1,
    lambda_partition=lambda_partition,
)

explainer = CDEAExplainer(
    model=model,
    unit_space=unit_space,
    selector=selector,
    base_evidence=base_evidence,
    allocator=allocator,
    objective=objective,
    interaction=interaction,
    normalize_evidence=True,
    device=device,
)

In [10]:
explanation = explainer.explain(x)
print({
    'suff': float(explanation.metrics['suff'].item()),
    'margin': float(explanation.metrics['margin'].item()),
    'overlap': float(explanation.metrics['overlap'].item()),
    'sparse': float(explanation.metrics['sparse'].item()),
})

out_path = REPO / 'examples' / 'out' / f'notebook_{evidence_mode}_{dataset}.png'
visualize_contrastive(
    sample_idx=0,
    x=x.detach().cpu(),
    explanation=explanation,
    unit_space=unit_space,
    class_names=class_names,
    evidence=explanation.extras.get('evidence'),
    probs=explanation.extras.get('probs'),
    out_path=out_path,
)
out_path

{'suff': 0.28148019313812256, 'margin': -0.1592216044664383, 'overlap': 0.00543671241030097, 'sparse': 0.23443052172660828}
Saved visualization to /Users/ssuresh/gambit/examples/out/notebook_gradcam_cifar10.png


PosixPath('/Users/ssuresh/gambit/examples/out/notebook_gradcam_cifar10.png')

In [11]:
import pandas as pd

ids0 = explanation.hypotheses.ids[0].detach().cpu()
valid0 = explanation.hypotheses.mask[0].detach().cpu()
split_shared = explanation.metrics.get('split_shared_only_probs_topm')
split_plus = explanation.metrics.get('split_shared_plus_unique_probs_topm')

rows = []
if isinstance(split_shared, torch.Tensor) and isinstance(split_plus, torch.Tensor):
    sh0 = split_shared[0].detach().cpu()
    pl0 = split_plus[0].detach().cpu()
    for k in range(int(valid0.sum().item())):
        cid = int(ids0[k].item())
        lbl = class_names[cid] if cid < len(class_names) else str(cid)
        rows.append({
            'rank': k,
            'class_id': cid,
            'label': lbl,
            'p_shared_only': float(sh0[k].item()),
            'p_shared_plus_unique': float(pl0[k].item()),
            'delta': float(pl0[k].item() - sh0[k].item()),
        })

pd.DataFrame(rows)

,rank,class_id,label,p_shared_only,p_shared_plus_unique,delta
0,0,6,frog,0.123012,0.123023,1.072884e-05
1,1,5,dog,0.121904,0.121908,3.673136e-06
2,2,7,horse,0.114122,0.114124,1.072884e-06
3,3,2,bird,0.102881,0.102880,-8.791685e-07
4,4,9,truck,0.099533,0.099532,-1.460314e-06


In [12]:
pair_shared = explanation.metrics.get('pairwise_margin_shared_only_logits_topm')
pair_plus = explanation.metrics.get('pairwise_margin_shared_plus_unique_logits_topm')
pair_delta = explanation.metrics.get('pairwise_margin_delta_logits_topm')

pair_rows = []
if isinstance(pair_shared, torch.Tensor) and isinstance(pair_plus, torch.Tensor) and isinstance(pair_delta, torch.Tensor):
    ps = pair_shared[0].detach().cpu()
    pp = pair_plus[0].detach().cpu()
    pd = pair_delta[0].detach().cpu()
    for k in range(int(valid0.sum().item())):
        for l in range(int(valid0.sum().item())):
            if k == l:
                continue
            cid_k = int(ids0[k].item())
            cid_l = int(ids0[l].item())
            pair_rows.append({
                'k_rank': k,
                'l_rank': l,
                'k_label': class_names[cid_k] if cid_k < len(class_names) else str(cid_k),
                'l_label': class_names[cid_l] if cid_l < len(class_names) else str(cid_l),
                'shared_only_margin_logit': float(ps[k, l].item()),
                'shared_plus_unique_margin_logit': float(pp[k, l].item()),
                'delta_margin_logit': float(pd[k, l].item()),
            })

pd.DataFrame(pair_rows).sort_values('delta_margin_logit', ascending=False)

AttributeError: 'Tensor' object has no attribute 'DataFrame'